Purpose
---
This script generates descriptive statistics for a Wikibase-based knowledge graph
through SPARQL queries. It computes distributions of items by class and category,
counts statements and qualifiers by property, and analyzes referenced statements
grouped by source provenance, exporting all results as CSV reports.

# Setup

In [1]:
import requests
import csv
import time

ENDPOINT = "effkg"
SOURCE_PROP = "P46"
SPARQL_ENDPOINT = f"https://{ENDPOINT}.wikibase.cloud/query/sparql"

HEADERS = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "statement-counter/1.0"
}

"""Execute a SPARQL query against the configured endpoint."""
def run_query(query):
    r = requests.get(
        SPARQL_ENDPOINT,
        params={"query": query, "format": "json"},
        headers=HEADERS,
        timeout=120
    )
    r.raise_for_status()
    return r.json()

"""Execute a SPARQL query with retry and exponential backoff support."""
def run_query_timeout(query, timeout=120, max_retries=4, backoff_base=2):
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(
                SPARQL_ENDPOINT,
                params={"query": query, "format": "json"},
                headers=HEADERS,
                timeout=timeout
            )

            if response.status_code >= 500:
                raise requests.HTTPError(
                    f"HTTP {response.status_code}: {response.text[:300]}",
                    response=response
                )

            response.raise_for_status()
            return response.json()

        except requests.RequestException as e:
            last_error = e
            if attempt == max_retries:
                break

            sleep_seconds = backoff_base ** (attempt - 1)
            print(f"  attempt {attempt}/{max_retries} failed: {e}")
            print(f"  retrying in {sleep_seconds}s...")
            time.sleep(sleep_seconds)

    raise last_error

# Items by type

In [ ]:
INSTANCE_OF = "P35"
SUBCLASS_OF = "P52"

# -------------------------------------------------------------------
# Retrieve all entities that are subclass of another entity
# -------------------------------------------------------------------

classes_query = f"""
PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?classId ?classLabel ?parentClassId ?parentClassLabel
WHERE {{
  ?class wdt:{SUBCLASS_OF} ?parentClass .

  OPTIONAL {{
    ?class rdfs:label ?classLabel .
    FILTER(LANG(?classLabel) = "en")
  }}

  OPTIONAL {{
    ?parentClass rdfs:label ?parentClassLabel .
    FILTER(LANG(?parentClassLabel) = "en")
  }}

  BIND(REPLACE(STR(?class), "^.*/entity/", "") AS ?classId)
  BIND(REPLACE(STR(?parentClass), "^.*/entity/", "") AS ?parentClassId)
}}
ORDER BY ?classId
"""

class_results = run_query(classes_query)

classes = []

for b in class_results["results"]["bindings"]:
    class_id = b["classId"]["value"]
    parent_class_id = b["parentClassId"]["value"]

    class_label = b.get("classLabel", {}).get("value", class_id)
    parent_class_label = b.get("parentClassLabel", {}).get("value", parent_class_id)

    classes.append(
        (
            class_id,
            class_label,
            parent_class_id,
            parent_class_label,
        )
    )

print(f"Classes found: {len(classes):,}")

# -------------------------------------------------------------------
# Count instances for each class
# -------------------------------------------------------------------

rows = []
total_instances = 0

for class_id, class_label, parent_class_id, parent_class_label in classes:
    q = f"""
    PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>
    PREFIX wd: <https://{ENDPOINT}.wikibase.cloud/entity/>

    SELECT (COUNT(DISTINCT ?item) AS ?count)
    WHERE {{
      ?item wdt:{INSTANCE_OF} wd:{class_id} .
    }}
    """

    try:
        res = run_query(q)
        count = int(res["results"]["bindings"][0]["count"]["value"])

        if count > 0:
            rows.append(
                (
                    class_id,
                    class_label,
                    parent_class_id,
                    parent_class_label,
                    count,
                )
            )

            total_instances += count

    except Exception as e:
        print(f"Error with {class_id} ({class_label}): {e}")

# -------------------------------------------------------------------
# Sort descending by number of instances
# -------------------------------------------------------------------

rows.sort(key=lambda x: x[4], reverse=True)

# -------------------------------------------------------------------
# Pretty-print results table
# -------------------------------------------------------------------

def print_table(rows):
    headers = [
        "Class",
        "Class label",
        "Parent",
        "Parent label",
        "Instances"
    ]

    table_rows = [
        [
            class_id,
            class_label,
            parent_class_id,
            parent_class_label,
            f"{count:,}",
        ]
        for class_id, class_label, parent_class_id, parent_class_label, count in rows
    ]

    all_rows = [headers] + table_rows

    col_widths = [
        max(len(str(row[i])) for row in all_rows)
        for i in range(len(headers))
    ]

    separator = "-+-".join("-" * width for width in col_widths)

    print("\n=== ITEMS BY CLASS ===\n")

    print(
        " | ".join(
            headers[i].ljust(col_widths[i])
            for i in range(len(headers))
        )
    )

    print(separator)

    for row in table_rows:
        print(
            " | ".join(
                str(row[i]).ljust(col_widths[i])
                for i in range(len(headers))
            )
        )

# -------------------------------------------------------------------
# Display results
# -------------------------------------------------------------------

print_table(rows)

print(f"\nTOTAL INSTANCE RELATIONS: {total_instances:,}")

# -------------------------------------------------------------------
# Export results to CSV
# -------------------------------------------------------------------

with open("items_by_class.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")

    writer.writerow(
        [
            "Class",
            "ClassLabel",
            "ParentClass",
            "ParentClassLabel",
            "NumberOfInstances",
        ]
    )

    writer.writerows(rows)

    writer.writerow([])
    writer.writerow(["TOTAL", "", "", "", total_instances])

print("\nCSV exported: items_by_class.csv")

Classes found: 13

=== ITEMS BY CLASS ===

Class  | Class label                                  | Parent | Parent label                 | Instances
-------+----------------------------------------------+--------+------------------------------+----------
Q24    | Fishing Vessel                               | Q23    | Vessel                       | 185,025  
Q398   | Vessel supporting fishing related activities | Q23    | Vessel                       | 3,145    
Q62    | Fishing Port                                 | Q488   | Port                         | 1,997    
Q13    | Fishing Area                                 | Q28608 | Maritime administrative unit | 114      
Q138   | Maritimal District                           | Q28608 | Maritime administrative unit | 108      
Q63    | Maritimal Province                           | Q28608 | Maritime administrative unit | 30       
Q28610 | Spanish Autonomous Community                 | Q379   | Land administrative unit     | 17       
Q47

## Categories

In [ ]:
INSTANCE_OF = "P35"
CATEGORY_OF = "P50"
CATEGORY_QID = "Q473"

ITEM_CATEGORY_PROPERTIES = [
    INSTANCE_OF,
    "P25",
]

query = f"""
PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>
PREFIX wd: <https://{ENDPOINT}.wikibase.cloud/entity/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?categoryId ?categoryLabel
       ?categoryOfId ?categoryOfLabel
       ?subCategoryId ?subCategoryLabel
       (COUNT(DISTINCT ?item) AS ?numberOfItems)
WHERE {{
  ?category wdt:{INSTANCE_OF} wd:{CATEGORY_QID} .

  OPTIONAL {{
    ?category wdt:{CATEGORY_OF} ?categoryOf .
    BIND(REPLACE(STR(?categoryOf), "^.*/entity/", "") AS ?categoryOfId)

    OPTIONAL {{
      ?categoryOf rdfs:label ?categoryOfLabel .
      FILTER(LANG(?categoryOfLabel) = "en")
    }}
  }}

  ?subCategory wdt:{INSTANCE_OF} ?category .

  VALUES ?itemCategoryProperty {{
    {" ".join("wdt:" + p for p in ITEM_CATEGORY_PROPERTIES)}
  }}

  OPTIONAL {{
    ?item ?itemCategoryProperty ?subCategory .
  }}

  BIND(REPLACE(STR(?category), "^.*/entity/", "") AS ?categoryId)
  BIND(REPLACE(STR(?subCategory), "^.*/entity/", "") AS ?subCategoryId)

  OPTIONAL {{
    ?category rdfs:label ?categoryLabel .
    FILTER(LANG(?categoryLabel) = "en")
  }}

  OPTIONAL {{
    ?subCategory rdfs:label ?subCategoryLabel .
    FILTER(LANG(?subCategoryLabel) = "en")
  }}
}}
GROUP BY ?categoryId ?categoryLabel ?categoryOfId ?categoryOfLabel ?subCategoryId ?subCategoryLabel
ORDER BY ?categoryLabel DESC(?numberOfItems)
"""

results = run_query(query)

rows = []

for b in results["results"]["bindings"]:
    category_id = b["categoryId"]["value"]
    category_label = b.get("categoryLabel", {}).get("value", category_id)

    category_of_id = b.get("categoryOfId", {}).get("value", "")
    category_of_label = b.get("categoryOfLabel", {}).get("value", category_of_id)

    subcategory_id = b["subCategoryId"]["value"]
    subcategory_label = b.get("subCategoryLabel", {}).get("value", subcategory_id)

    count = int(b["numberOfItems"]["value"])

    rows.append(
        (
            category_id,
            category_label,
            category_of_id,
            category_of_label,
            subcategory_id,
            subcategory_label,
            count,
        )
    )


def print_table(rows):
    headers = [
        "Category",
        "Category label",
        "Category of",
        "Category of label",
        "Subcategory",
        "Subcategory label",
        "Items",
    ]

    table_rows = [
        [
            category_id,
            category_label,
            category_of_id,
            category_of_label,
            subcategory_id,
            subcategory_label,
            f"{count:,}",
        ]
        for (
            category_id,
            category_label,
            category_of_id,
            category_of_label,
            subcategory_id,
            subcategory_label,
            count,
        ) in rows
    ]

    all_rows = [headers] + table_rows

    col_widths = [
        max(len(str(row[i])) for row in all_rows)
        for i in range(len(headers))
    ]

    separator = "-+-".join("-" * width for width in col_widths)

    print("\n=== CATEGORY ITEMS: SECOND LEVEL ===\n")
    print(" | ".join(headers[i].ljust(col_widths[i]) for i in range(len(headers))))
    print(separator)

    for row in table_rows:
        print(" | ".join(str(row[i]).ljust(col_widths[i]) for i in range(len(headers))))


print_table(rows)

total_subcategories = len(rows)
total_items = sum(row[6] for row in rows)

print(f"\nTOTAL SUBCATEGORIES: {total_subcategories:,}")
print(f"TOTAL ITEM RELATIONS: {total_items:,}")

with open("category_items_second_level.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")

    writer.writerow(
        [
            "Category",
            "CategoryLabel",
            "CategoryOf",
            "CategoryOfLabel",
            "Subcategory",
            "SubcategoryLabel",
            "NumberOfItems",
        ]
    )

    writer.writerows(rows)

    writer.writerow([])
    writer.writerow(["TOTAL SUBCATEGORIES", total_subcategories, "", "", "", "", ""])
    writer.writerow(["TOTAL ITEM RELATIONS", "", "", "", "", "", total_items])

print("\nCSV exported: category_items_second_level.csv")


=== CATEGORY ITEMS: SECOND LEVEL ===

Category | Category label        | Category of | Category of label | Subcategory | Subcategory label                                              | Items  
---------+-----------------------+-------------+-------------------+-------------+----------------------------------------------------------------+--------
Q472     | Fishing Gear Category | Q469        | Fishing Gear      | Q511        | Set gillnets (anchored)                                        | 106,012
Q472     | Fishing Gear Category | Q469        | Fishing Gear      | Q529        | Set longlines                                                  | 60,998 
Q472     | Fishing Gear Category | Q469        | Fishing Gear      | Q546        | Gear not known                                                 | 51,925 
Q472     | Fishing Gear Category | Q469        | Fishing Gear      | Q515        | Trammel nets                                                   | 39,274 
Q472     | Fishing Gear C

# Statements by property

In [ ]:
# Get all property IDs and labels
properties_query = """
PREFIX wikibase: <http://wikiba.se/ontology#>

SELECT ?propId ?propertyLabel
WHERE {
  ?property a wikibase:Property ;
            wikibase:claim ?p ;
            rdfs:label ?propertyLabel .
  FILTER(LANG(?propertyLabel) = "en")
  BIND(REPLACE(STR(?property), "^.*/entity/", "") AS ?propId)
}
ORDER BY ?propId
"""

prop_results = run_query(properties_query)
properties = [
    (b["propId"]["value"], b["propertyLabel"]["value"])
    for b in prop_results["results"]["bindings"]
]

rows = []
total = 0

for prop_id, prop_label in properties:
    q = f"""
    PREFIX p: <https://{ENDPOINT}.wikibase.cloud/prop/>

    SELECT (COUNT(?statement) AS ?count)
    WHERE {{
      ?item p:{prop_id} ?statement .
    }}
    """
    try:
        res = run_query(q)
        count = int(res["results"]["bindings"][0]["count"]["value"])
        if count > 0:
            rows.append((prop_id, prop_label, count))
            total += count
    except Exception as e:
        print(f"Error with {prop_id} ({prop_label}): {e}")

rows.sort(key=lambda x: x[2], reverse=True)

print("=== STATEMENTS BY PROPERTY ===")
for prop_id, prop_label, count in rows:
    print(f"{prop_id}\t{prop_label}\t{count}")

print(f"\nTOTAL STATEMENTS: {total}")

with open("statements_by_property.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow(["Property", "Label", "NumberOfStatements"])
    writer.writerows(rows)
    writer.writerow([])
    writer.writerow(["TOTAL", "", total])

=== STATEMENTS BY PROPERTY ===
P25	has fishing gear category	371013
P23	main engine power	213317
P35	instance of	190592
P12	CFR	188177
P51	has vessel category	187575
P55	country of registration	185685
P9	external marking	185617
P59	segment	184804
P10	registration number	183401
P29	entry into service date	183396
P37	name	182619
P16	LOA	181359
P27	hull material	175595
P30	date of construction	160307
P20	GT Tonnage	153629
P21	other tonnage	126242
P11	registration place	124760
P28	on board equipment	115614
P17	LBP	77697
P58	date of retirement	71896
P57	date of destruction	42551
P26	code	32992
P32	status on national registry	30730
P49	IMO	30529
P31	registration on national registry	30527
P56	administration responsible of registration	30527
P41	operates at	23829
P15	MMSI	20309
P24	auxiliar engine power	9499
P45	number of vessels	5177
P14	UVI	3868
P42	coordinates	1927
P3	belongs to administrative unit	430
P22	safety tonnage GTs	338
P7	has ports	295
P60	administrated by	155
P43	contains admini

# Qualifiers by property

In [ ]:
# Retrieve properties that can act as qualifiers
qualifier_props_query = """
PREFIX wikibase: <http://wikiba.se/ontology#>

SELECT ?propId ?propertyLabel
WHERE {
  ?property a wikibase:Property ;
            wikibase:qualifier ?pqProp ;
            rdfs:label ?propertyLabel .
  FILTER(LANG(?propertyLabel) = "en")
  BIND(REPLACE(STR(?property), "^.*/entity/", "") AS ?propId)
}
ORDER BY ?propId
"""

results = run_query(qualifier_props_query)
props = [
    (b["propId"]["value"], b["propertyLabel"]["value"])
    for b in results["results"]["bindings"]
]

rows = []
total = 0

for prop_id, prop_label in props:
    q = f"""
    PREFIX pq: <https://{ENDPOINT}.wikibase.cloud/prop/qualifier/>

    SELECT (COUNT(DISTINCT ?statement) AS ?count)
    WHERE {{
      ?statement pq:{prop_id} ?value .
    }}
    """
    try:
        res = run_query(q)
        count = int(res["results"]["bindings"][0]["count"]["value"])
        if count > 0:
            rows.append((prop_id, prop_label, count))
            total += count
            print(f"{prop_id}\t{prop_label}\t{count}")
    except Exception as e:
        print(f"Error with {prop_id} ({prop_label}): {e}")

rows.sort(key=lambda x: x[2], reverse=True)

print("\n=== QUALIFIERS BY PROPERTY ===")
for prop_id, prop_label, count in rows:
    print(f"{prop_id}\t{prop_label}\t{count}")

print(f"\nTOTAL QUALIFIED STATEMENTS COUNTED: {total}")

P13	IRCS	52858
P32	status on national registry	3
P33	since	30730
P44	date	5180
P47	start date	129
P48	end date	57
P55	country of registration	12682

=== QUALIFIERS BY PROPERTY ===
P13	IRCS	52858
P33	since	30730
P55	country of registration	12682
P44	date	5180
P47	start date	129
P48	end date	57
P32	status on national registry	3

TOTAL QUALIFIED STATEMENTS COUNTED: 101639


# References by source value

In [ ]:
SOURCES = [
    "https://webgate.ec.europa.eu/fleet-europa",
    "https://servicio.pesca.mapama.es/CENSO/ConsultaBuqueRegistro/Buques/Search",
    "https://signa.ign.es/signa/",
    "https://boe.es/buscar/pdf/2007/BOE-A-2007-10951-consolidado.pdf",
    "https://www.mapa.gob.es/es/pesca/temas/registro-flota/informacion-sobre-flota-pesquera",
    "https://openknowledge.fao.org/handle/20.500.14283/cb5201en",
    "https://openknowledge.fao.org/items/4de1cb7a-34e3-4645-ae94-0cfc3ef4ad70",
    "https://openknowledge.fao.org/server/api/core/bitstreams/acee638a-d119-4fc6-8512-d7d953b14cd8/content",
    "https://app.hubocean.earth/catalog",
    "https://msi.nga.mil",
    "https://www.data.gouv.fr/api/1/datasets/r/60fe965d-5888-493b-9321-24bc3b1f84db",
    "https://www.fao.org/fishery/en/area",
    "https://www.boe.es/eli/es/rd/2007/05/18/638/con",
]

"""Count statements referencing a specific source URL."""
def count_statements_for_source(source_url):
    query = f"""
    PREFIX prov: <http://www.w3.org/ns/prov#>
    PREFIX pr: <https://{ENDPOINT}.wikibase.cloud/prop/reference/>

    SELECT (COUNT(DISTINCT ?statement) AS ?count)
    WHERE {{
      ?statement prov:wasDerivedFrom ?reference .
      ?reference pr:{SOURCE_PROP} <{source_url}> .
    }}
    """

    result = run_query_timeout(query)
    bindings = result.get("results", {}).get("bindings", [])
    if not bindings:
        return 0

    return int(bindings[0]["count"]["value"])

"""Generate statistics for statements grouped by reference source."""
def main():
    rows = []
    total = 0

    print("=== REFERENCES BY SOURCE VALUE ===")

    for idx, source in enumerate(SOURCES, start=1):
        print(f"[{idx}/{len(SOURCES)}] {source}")

        try:
            count = count_statements_for_source(source)
            rows.append({
                "source": source,
                "numberOfStatements": count,
                "status": "OK",
                "error": ""
            })
            total += count
            print(f"  -> {count}")

        except Exception as e:
            rows.append({
                "source": source,
                "numberOfStatements": "",
                "status": "ERROR",
                "error": str(e)
            })
            print(f"  -> ERROR: {e}")

        time.sleep(1)

    print(f"\nTOTAL REFERENCED STATEMENTS: {total}")

    rows_sorted = sorted(
        rows,
        key=lambda r: r["numberOfStatements"] if isinstance(r["numberOfStatements"], int) else -1,
        reverse=True
    )

    with open("references_by_source.csv", "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f, delimiter=";")
        writer.writerow(["source", "numberOfStatements", "status", "error"])

        for row in rows_sorted:
            writer.writerow([
                row["source"],
                row["numberOfStatements"],
                row["status"],
                row["error"]
            ])

        writer.writerow([])
        writer.writerow(["TOTAL", total, "", ""])

    print("\nCSV generated: references_by_source.csv")

if __name__ == "__main__":
    main()

=== REFERENCES BY SOURCE VALUE ===
[1/13] https://webgate.ec.europa.eu/fleet-europa
  -> 3448294
[2/13] https://servicio.pesca.mapama.es/CENSO/ConsultaBuqueRegistro/Buques/Search
  -> 570603
[3/13] https://signa.ign.es/signa/
  -> 193
[4/13] https://boe.es/buscar/pdf/2007/BOE-A-2007-10951-consolidado.pdf
  -> 293
[5/13] https://www.mapa.gob.es/es/pesca/temas/registro-flota/informacion-sobre-flota-pesquera
  -> 5158
[6/13] https://openknowledge.fao.org/handle/20.500.14283/cb5201en
  -> 189
[7/13] https://openknowledge.fao.org/items/4de1cb7a-34e3-4645-ae94-0cfc3ef4ad70
  -> 367
[8/13] https://openknowledge.fao.org/server/api/core/bitstreams/acee638a-d119-4fc6-8512-d7d953b14cd8/content
  -> 9
[9/13] https://app.hubocean.earth/catalog
  -> 4
[10/13] https://msi.nga.mil
  -> 995
[11/13] https://www.data.gouv.fr/api/1/datasets/r/60fe965d-5888-493b-9321-24bc3b1f84db
  -> 6313
[12/13] https://www.fao.org/fishery/en/area
  -> 569
[13/13] https://www.boe.es/eli/es/rd/2007/05/18/638/con
  -> 858


# Provenance coverage

In [2]:
# Get all properties with claims

properties_query = """
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?propId ?propertyLabel
WHERE {
  ?property a wikibase:Property ;
            wikibase:claim ?claimProp .

  OPTIONAL {
    ?property rdfs:label ?propertyLabel .
    FILTER(LANG(?propertyLabel) = "en")
  }

  BIND(REPLACE(STR(?property), "^.*/entity/", "") AS ?propId)
}
ORDER BY ?propId
"""

prop_results = run_query(properties_query)

properties = [
    (
        b["propId"]["value"],
        b.get("propertyLabel", {}).get("value", b["propId"]["value"])
    )
    for b in prop_results["results"]["bindings"]
]

rows = []
total_statements = 0
total_statements_with_reference = 0

print("\n=== PROVENANCE COVERAGE BY PROPERTY ===\n")

for idx, (prop_id, prop_label) in enumerate(properties, start=1):
    print(f"[{idx}/{len(properties)}] {prop_id} {prop_label}")

    query = f"""
    PREFIX p: <https://{ENDPOINT}.wikibase.cloud/prop/>
    PREFIX prov: <http://www.w3.org/ns/prov#>

    SELECT ?totalStatements ?statementsWithReference
    WHERE {{
      {{
        SELECT (COUNT(?statement) AS ?totalStatements)
        WHERE {{
          ?entity p:{prop_id} ?statement .
        }}
      }}

      {{
        SELECT (COUNT(DISTINCT ?statement) AS ?statementsWithReference)
        WHERE {{
          ?entity p:{prop_id} ?statement .
          ?statement prov:wasDerivedFrom ?reference .
        }}
      }}
    }}
    """

    try:
        results = run_query_timeout(query)
        b = results["results"]["bindings"][0]

        property_total = int(b["totalStatements"]["value"])
        property_with_reference = int(b["statementsWithReference"]["value"])
        property_without_reference = property_total - property_with_reference

        property_percent_with_reference = (
            property_with_reference / property_total * 100
            if property_total else 0
        )

        if property_total > 0:
            rows.append((
                prop_id,
                prop_label,
                property_total,
                property_with_reference,
                property_without_reference,
                property_percent_with_reference,
            ))

            total_statements += property_total
            total_statements_with_reference += property_with_reference

        print(
            f"  -> total={property_total:,}; "
            f"with reference={property_with_reference:,}; "
            f"without reference={property_without_reference:,}; "
            f"coverage={property_percent_with_reference:.3f}%"
        )

    except Exception as e:
        print(f"  -> ERROR: {e}")
        rows.append((prop_id, prop_label, "", "", "", ""))

    time.sleep(0.5)

total_statements_without_reference = (
    total_statements - total_statements_with_reference
)

percent_with_reference = (
    total_statements_with_reference / total_statements * 100
    if total_statements else 0
)

rows_valid = [r for r in rows if r[2] != ""]
rows_valid.sort(key=lambda x: x[5])

print("\n=== PROVENANCE COVERAGE ===\n")
print("Total statements | Statements with reference | Statements without reference | % with reference")
print("-----------------+---------------------------+------------------------------+-----------------")
print(
    f"{total_statements:,} | "
    f"{total_statements_with_reference:,} | "
    f"{total_statements_without_reference:,} | "
    f"{percent_with_reference:.3f}"
)

print("\n=== PROVENANCE COVERAGE BY PROPERTY ===\n")
print("Property | Label | Total | With reference | Without reference | % with reference")
print("---------+-------+-------+----------------+-------------------+-----------------")

for prop_id, prop_label, prop_total, prop_with_ref, prop_without_ref, prop_percent in rows_valid:
    print(
        f"{prop_id}\t{prop_label}\t{prop_total:,}\t"
        f"{prop_with_ref:,}\t{prop_without_ref:,}\t{prop_percent:.3f}"
    )

with open("provenance_coverage.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow([
        "TotalStatements",
        "StatementsWithReference",
        "StatementsWithoutReference",
        "PercentWithReference"
    ])
    writer.writerow([
        total_statements,
        total_statements_with_reference,
        total_statements_without_reference,
        percent_with_reference
    ])

with open("provenance_coverage_by_property.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow([
        "Property",
        "PropertyLabel",
        "TotalStatements",
        "StatementsWithReference",
        "StatementsWithoutReference",
        "PercentWithReference"
    ])
    writer.writerows(rows_valid)

print("\nCSV exported: provenance_coverage.csv")
print("CSV exported: provenance_coverage_by_property.csv")


=== PROVENANCE COVERAGE BY PROPERTY ===

[1/60] P1 defined by
  -> total=3; with reference=3; without reference=0; coverage=100.000%
[2/60] P10 registration number
  -> total=183,401; with reference=183,401; without reference=0; coverage=100.000%
[3/60] P11 registration place
  -> total=124,760; with reference=124,760; without reference=0; coverage=100.000%
[4/60] P12 CFR
  -> total=188,177; with reference=188,177; without reference=0; coverage=100.000%
[5/60] P13 IRCS
  -> total=0; with reference=0; without reference=0; coverage=0.000%
[6/60] P14 UVI
  -> total=3,868; with reference=3,868; without reference=0; coverage=100.000%
[7/60] P15 MMSI
  -> total=20,309; with reference=20,309; without reference=0; coverage=100.000%
[8/60] P16 LOA
  -> total=181,359; with reference=181,359; without reference=0; coverage=100.000%
[9/60] P17 LBP
  -> total=77,697; with reference=77,697; without reference=0; coverage=100.000%
[10/60] P18 has parent category
  -> total=107; with reference=107; wit

# Identifier uniqueness validation

In [ ]:
IDENTIFIER_PROPERTIES = [
    ("CFR", "P12"),
    ("IMO", "P49"),
    ("UVI", "P14"),
    ("MMSI", "P15"),
    ("Registration number", "P10"),
    ("External marking", "P9"),
]

PLACEHOLDER_VALUE = "-"

rows = []

for identifier_label, prop_id in IDENTIFIER_PROPERTIES:

    query = f"""
    PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>

    SELECT ?sourceLevelEntities ?sourceLevelValues
           ?sourceLevelDuplicatedValues ?sourceLevelAffectedEntities
           ?placeholderFilteredEntities ?placeholderFilteredValues
           ?placeholderFilteredDuplicatedValues ?placeholderFilteredAffectedEntities
           (((?sourceLevelValues - ?sourceLevelDuplicatedValues) / ?sourceLevelValues) * 100 AS ?sourceLevelValueUniquenessRate)
           (((?sourceLevelEntities - ?sourceLevelAffectedEntities) / ?sourceLevelEntities) * 100 AS ?sourceLevelEntityUniquenessRate)
           (((?placeholderFilteredValues - ?placeholderFilteredDuplicatedValues) / ?placeholderFilteredValues) * 100 AS ?placeholderFilteredValueUniquenessRate)
           (((?placeholderFilteredEntities - ?placeholderFilteredAffectedEntities) / ?placeholderFilteredEntities) * 100 AS ?placeholderFilteredEntityUniquenessRate)
    WHERE {{
      {{
        SELECT
          (COUNT(?entity) AS ?sourceLevelEntities)
          (COUNT(DISTINCT ?id) AS ?sourceLevelValues)
        WHERE {{
          ?entity wdt:{prop_id} ?id .
        }}
      }}

      {{
        SELECT
          (COUNT(?id) AS ?sourceLevelDuplicatedValues)
          (SUM(?nEntities) AS ?sourceLevelAffectedEntities)
        WHERE {{
          {{
            SELECT ?id (COUNT(DISTINCT ?entity) AS ?nEntities)
            WHERE {{
              ?entity wdt:{prop_id} ?id .
            }}
            GROUP BY ?id
            HAVING (COUNT(DISTINCT ?entity) > 1)
          }}
        }}
      }}

      {{
        SELECT
          (COUNT(?entity) AS ?placeholderFilteredEntities)
          (COUNT(DISTINCT ?id) AS ?placeholderFilteredValues)
        WHERE {{
          ?entity wdt:{prop_id} ?id .
          FILTER(STR(?id) != "{PLACEHOLDER_VALUE}")
        }}
      }}

      {{
        SELECT
          (COUNT(?id) AS ?placeholderFilteredDuplicatedValues)
          (SUM(?nEntities) AS ?placeholderFilteredAffectedEntities)
        WHERE {{
          {{
            SELECT ?id (COUNT(DISTINCT ?entity) AS ?nEntities)
            WHERE {{
              ?entity wdt:{prop_id} ?id .
              FILTER(STR(?id) != "{PLACEHOLDER_VALUE}")
            }}
            GROUP BY ?id
            HAVING (COUNT(DISTINCT ?entity) > 1)
          }}
        }}
      }}
    }}
    """

    try:
        results = run_query(query)
        b = results["results"]["bindings"][0]

        rows.append((
            identifier_label,
            prop_id,

            int(b["sourceLevelEntities"]["value"]),
            int(b["sourceLevelValues"]["value"]),
            int(b["sourceLevelDuplicatedValues"]["value"]),
            int(b["sourceLevelAffectedEntities"]["value"]),
            float(b["sourceLevelValueUniquenessRate"]["value"]),
            float(b["sourceLevelEntityUniquenessRate"]["value"]),

            int(b["placeholderFilteredEntities"]["value"]),
            int(b["placeholderFilteredValues"]["value"]),
            int(b["placeholderFilteredDuplicatedValues"]["value"]),
            int(b["placeholderFilteredAffectedEntities"]["value"]),
            float(b["placeholderFilteredValueUniquenessRate"]["value"]),
            float(b["placeholderFilteredEntityUniquenessRate"]["value"]),
        ))

    except Exception as e:
        print(f"Error with {identifier_label} ({prop_id}): {e}")
        rows.append((
            identifier_label, prop_id,
            "", "", "", "", "", "",
            "", "", "", "", "", ""
        ))

print("\n=== IDENTIFIER UNIQUENESS VALIDATION ===\n")
print(
    "Identifier | Property | "
    "Source-level entities | Source-level values | Source-level duplicated values | "
    "Source-level affected entities | Source-level entity uniqueness % | "
    "Placeholder-filtered entities | Placeholder-filtered values | "
    "Placeholder-filtered duplicated values | Placeholder-filtered affected entities | "
    "Placeholder-filtered entity uniqueness %"
)
print(
    "-----------+----------+-----------------------+---------------------+-------------------------------+"
    "-------------------------------+----------------------------------+-------------------------------+"
    "-----------------------------+---------------------------------------+--------------------------------------+"
    "------------------------------------------"
)

for (
    identifier_label,
    prop_id,

    source_level_entities,
    source_level_values,
    source_level_duplicated_values,
    source_level_affected_entities,
    source_level_value_uniqueness_rate,
    source_level_entity_uniqueness_rate,

    placeholder_filtered_entities,
    placeholder_filtered_values,
    placeholder_filtered_duplicated_values,
    placeholder_filtered_affected_entities,
    placeholder_filtered_value_uniqueness_rate,
    placeholder_filtered_entity_uniqueness_rate,
) in rows:

    if source_level_entities == "":
        print(f"{identifier_label}\t{prop_id}\tERROR")
    else:
        print(
            f"{identifier_label}\t{prop_id}\t"
            f"{source_level_entities:,}\t"
            f"{source_level_values:,}\t"
            f"{source_level_duplicated_values:,}\t"
            f"{source_level_affected_entities:,}\t"
            f"{source_level_entity_uniqueness_rate:.3f}\t"
            f"{placeholder_filtered_entities:,}\t"
            f"{placeholder_filtered_values:,}\t"
            f"{placeholder_filtered_duplicated_values:,}\t"
            f"{placeholder_filtered_affected_entities:,}\t"
            f"{placeholder_filtered_entity_uniqueness_rate:.3f}"
        )

with open("identifier_uniqueness.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow([
        "Identifier",
        "Property",

        "SourceLevelEntitiesWithIdentifier",
        "SourceLevelDistinctValues",
        "SourceLevelDuplicatedValues",
        "SourceLevelAffectedEntities",
        "SourceLevelValueUniquenessRate",
        "SourceLevelEntityUniquenessRate",

        "PlaceholderFilteredEntitiesWithIdentifier",
        "PlaceholderFilteredDistinctValues",
        "PlaceholderFilteredDuplicatedValues",
        "PlaceholderFilteredAffectedEntities",
        "PlaceholderFilteredValueUniquenessRate",
        "PlaceholderFilteredEntityUniquenessRate"
    ])
    writer.writerows(rows)

print("\nCSV exported: identifier_uniqueness.csv")


=== IDENTIFIER UNIQUENESS VALIDATION ===

Identifier | Property | Source-level entities | Source-level values | Source-level duplicated values | Source-level affected entities | Source-level entity uniqueness % | Placeholder-filtered entities | Placeholder-filtered values | Placeholder-filtered duplicated values | Placeholder-filtered affected entities | Placeholder-filtered entity uniqueness %
-----------+----------+-----------------------+---------------------+-------------------------------+-------------------------------+----------------------------------+-------------------------------+-----------------------------+---------------------------------------+--------------------------------------+------------------------------------------
CFR	P12	188,177	185,297	3	2,883	98.468	185,298	185,296	2	4	99.998
IMO	P49	30,529	1,136	2	29,395	3.715	1,136	1,135	1	2	99.824
UVI	P14	3,868	3,866	2	4	99.897	3,868	3,866	2	4	99.897
MMSI	P15	20,309	20,290	19	38	99.813	20,309	20,290	19	38	99.813
Registr

# Broken Wikibase item link validation

In [ ]:
# First retrieve only WikibaseItem properties.
query = """
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?propId ?propertyLabel
WHERE {
  ?property a wikibase:Property ;
            wikibase:statementProperty ?statementProp ;
            wikibase:propertyType wikibase:WikibaseItem .

  OPTIONAL {
    ?property rdfs:label ?propertyLabel .
    FILTER(LANG(?propertyLabel) = "en")
  }

  BIND(REPLACE(STR(?property), "^.*/entity/", "") AS ?propId)
}
ORDER BY ?propId
"""

results = run_query(query)

wikibase_item_properties = []

for b in results["results"]["bindings"]:
    prop_id = b["propId"]["value"]
    prop_label = b.get("propertyLabel", {}).get("value", prop_id)
    wikibase_item_properties.append((prop_id, prop_label))

print(f"WikibaseItem properties found: {len(wikibase_item_properties):,}")

rows = []
total_item_links = 0
total_broken_links = 0

for idx, (prop_id, prop_label) in enumerate(wikibase_item_properties, start=1):
    print(f"[{idx}/{len(wikibase_item_properties)}] {prop_id} {prop_label}")

    query = f"""
PREFIX p: <https://{ENDPOINT}.wikibase.cloud/prop/>
PREFIX ps: <https://{ENDPOINT}.wikibase.cloud/prop/statement/>
PREFIX wikibase: <http://wikiba.se/ontology#>

SELECT ?totalItemLinks ?brokenItemLinks
WHERE {{
  {{
    SELECT (COUNT(?value) AS ?totalItemLinks)
    WHERE {{
      ?entity p:{prop_id} ?statement .
      ?statement ps:{prop_id} ?value .
    }}
  }}

  {{
    SELECT (COUNT(?value) AS ?brokenItemLinks)
    WHERE {{
      ?entity p:{prop_id} ?statement .
      ?statement ps:{prop_id} ?value .

      FILTER NOT EXISTS {{
        ?value wikibase:statements ?nStatements .
      }}
    }}
  }}
}}
    """

    try:
        res = run_query_timeout(query)
        b = res["results"]["bindings"][0]

        total = int(b["totalItemLinks"]["value"])
        broken = int(b["brokenItemLinks"]["value"])
        percent = (broken / total * 100) if total else 0

        if total > 0:
            rows.append((prop_id, prop_label, total, broken, percent))
            total_item_links += total
            total_broken_links += broken

        print(f"  -> total={total:,}; broken={broken:,}; broken%={percent:.3f}")

    except Exception as e:
        print(f"  -> ERROR: {e}")
        rows.append((prop_id, prop_label, "", "", ""))

    time.sleep(0.5)

rows_valid = [r for r in rows if r[2] != ""]
rows_valid.sort(key=lambda x: x[3], reverse=True)

print("\n=== BROKEN WIKIBASE ITEM LINKS ===\n")
print("Property | Label | Item links | Broken links | % broken")
print("---------+-------+------------+--------------+---------")

for prop_id, prop_label, total, broken, percent in rows_valid:
    print(f"{prop_id}\t{prop_label}\t{total:,}\t{broken:,}\t{percent:.3f}")

print(f"\nTOTAL ITEM LINKS: {total_item_links:,}")
print(f"TOTAL BROKEN LINKS: {total_broken_links:,}")
print(
    f"OVERALL BROKEN LINK RATE: "
    f"{(total_broken_links / total_item_links * 100 if total_item_links else 0):.3f}%"
)

with open("broken_wikibase_item_links.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow([
        "Property",
        "PropertyLabel",
        "TotalItemLinks",
        "BrokenItemLinks",
        "PercentBroken"
    ])
    writer.writerows(rows_valid)
    writer.writerow([])
    writer.writerow([
        "TOTAL",
        "",
        total_item_links,
        total_broken_links,
        (total_broken_links / total_item_links * 100 if total_item_links else 0)
    ])

print("\nCSV exported: broken_wikibase_item_links.csv")

WikibaseItem properties found: 18
[1/18] P11 registration place
  -> total=124,760; broken=0; broken%=0.000
[2/18] P18 has parent category
  -> total=107; broken=0; broken%=0.000
[3/18] P25 has fishing gear category
  -> total=371,013; broken=0; broken%=0.000
[4/18] P27 hull material
  -> total=175,595; broken=0; broken%=0.000
[5/18] P28 on board equipment
  -> total=115,614; broken=0; broken%=0.000
[6/18] P3 belongs to administrative unit
  -> total=430; broken=0; broken%=0.000
[7/18] P35 instance of
  -> total=190,592; broken=0; broken%=0.000
[8/18] P36 division of
  -> total=92; broken=0; broken%=0.000
[9/18] P38 subarea of
  -> total=21; broken=0; broken%=0.000
[10/18] P39 has divisions
  -> total=92; broken=0; broken%=0.000
[11/18] P40 has subareas
  -> total=19; broken=0; broken%=0.000
[12/18] P41 operates at
  -> total=23,829; broken=0; broken%=0.000
[13/18] P43 contains administrative unit
  -> total=136; broken=0; broken%=0.000
[14/18] P50 category of
  -> total=4; broken=0; b

# Domain plausibility validation

In [3]:
PLAUSIBILITY_CHECKS = [
    (
        "LOA greater than zero",
        f"""
        PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>
        SELECT (COUNT(?vessel) AS ?total) (SUM(IF(?loa <= 0, 1, 0)) AS ?anomalies)
        WHERE {{
          ?vessel wdt:P16 ?loa .
        }}
        """
    ),
    (
        "GT tonnage non-negative",
        f"""
        PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>
        SELECT (COUNT(?vessel) AS ?total) (SUM(IF(?gt < 0, 1, 0)) AS ?anomalies)
        WHERE {{
          ?vessel wdt:P20 ?gt .
        }}
        """
    ),
    (
        "Main engine power greater than 0",
        f"""
        PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>
        SELECT (COUNT(?vessel) AS ?total) (SUM(IF(?power < 0, 1, 0)) AS ?anomalies)
        WHERE {{
          ?vessel wdt:P23 ?power .
        }}
        """
    ),
    (
        "Construction date not after entry into service",
        f"""
        PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>
        SELECT (COUNT(?vessel) AS ?total)
               (SUM(IF(?constructionDate > ?entryDate, 1, 0)) AS ?anomalies)
        WHERE {{
          ?vessel wdt:P30 ?constructionDate ;
                  wdt:P29 ?entryDate .
        }}
        """
    ),
    (
        "Retirement date not before entry into service",
        f"""
        PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>
        SELECT (COUNT(?vessel) AS ?total)
               (SUM(IF(?retirementDate < ?entryDate, 1, 0)) AS ?anomalies)
        WHERE {{
          ?vessel wdt:P29 ?entryDate ;
                  wdt:P58 ?retirementDate .
        }}
        """
    ),
    (
        "Destruction date not before construction",
        f"""
        PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>
        SELECT (COUNT(?vessel) AS ?total)
               (SUM(IF(?destructionDate < ?constructionDate, 1, 0)) AS ?anomalies)
        WHERE {{
          ?vessel wdt:P30 ?constructionDate ;
                  wdt:P57 ?destructionDate .
        }}
        """
    ),
]

rows = []
total_checked = 0
total_anomalies = 0

for check_label, query in PLAUSIBILITY_CHECKS:
    try:
        results = run_query(query)
        b = results["results"]["bindings"][0]

        total = int(b["total"]["value"])
        anomalies = int(b["anomalies"]["value"]) if "anomalies" in b else 0
        valid = total - anomalies

        percent_valid = (valid / total * 100) if total else 0
        percent_anomalous = (anomalies / total * 100) if total else 0

        rows.append((
            check_label,
            total,
            valid,
            anomalies,
            percent_valid,
            percent_anomalous,
        ))

        total_checked += total
        total_anomalies += anomalies

    except Exception as e:
        print(f"Error with {check_label}: {e}")
        rows.append((check_label, "", "", "", "", ""))

overall_valid = total_checked - total_anomalies
overall_percent_valid = (overall_valid / total_checked * 100) if total_checked else 0
overall_percent_anomalous = (total_anomalies / total_checked * 100) if total_checked else 0

print("\n=== DOMAIN PLAUSIBILITY VALIDATION ===\n")
print("Check | Total checked | Valid | Anomalies | % valid | % anomalous")
print("------+---------------+-------+-----------+---------+------------")

for check_label, total, valid, anomalies, percent_valid, percent_anomalous in rows:
    if total == "":
        print(f"{check_label}\tERROR")
    else:
        print(
            f"{check_label}\t"
            f"{total:,}\t"
            f"{valid:,}\t"
            f"{anomalies:,}\t"
            f"{percent_valid:.3f}\t"
            f"{percent_anomalous:.3f}"
        )

print(f"\nTOTAL VALUES CHECKED: {total_checked:,}")
print(f"TOTAL VALID VALUES: {overall_valid:,}")
print(f"TOTAL DOMAIN ANOMALIES: {total_anomalies:,}")
print(f"OVERALL DOMAIN PLAUSIBILITY: {overall_percent_valid:.3f}%")
print(f"OVERALL DOMAIN ANOMALY RATE: {overall_percent_anomalous:.3f}%")

with open("domain_plausibility_validation.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow([
        "Check",
        "TotalChecked",
        "Valid",
        "DetectedAnomalies",
        "PercentValid",
        "PercentAnomalous"
    ])
    writer.writerows(rows)
    writer.writerow([])
    writer.writerow([
        "TOTAL",
        total_checked,
        overall_valid,
        total_anomalies,
        overall_percent_valid,
        overall_percent_anomalous
    ])

print("\nCSV exported: domain_plausibility_validation.csv")


=== DOMAIN PLAUSIBILITY VALIDATION ===

Check | Total checked | Valid | Anomalies | % valid | % anomalous
------+---------------+-------+-----------+---------+------------
LOA greater than zero	181,359	181,359	0	100.000	0.000
GT tonnage non-negative	153,629	153,629	0	100.000	0.000
Main engine power greater than 0	210,112	210,112	0	100.000	0.000
Construction date not after entry into service	161,022	160,937	85	99.947	0.053
Retirement date not before entry into service	70,535	70,417	118	99.833	0.167
Destruction date not before construction	32,642	32,642	0	100.000	0.000

TOTAL VALUES CHECKED: 809,299
TOTAL VALID VALUES: 809,096
TOTAL DOMAIN ANOMALIES: 203
OVERALL DOMAIN PLAUSIBILITY: 99.975%
OVERALL DOMAIN ANOMALY RATE: 0.025%

CSV exported: domain_plausibility_validation.csv


# Geospatial validation

In [ ]:
query = f"""
PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>

SELECT ?portId ?coord
WHERE {{
  ?port wdt:P42 ?coord .
  BIND(REPLACE(STR(?port), "^.*/entity/", "") AS ?portId)
}}
"""

results = run_query(query)

rows = []
invalid_rows = []

for b in results["results"]["bindings"]:
    port_id = b["portId"]["value"]
    coord = b["coord"]["value"]

    clean = coord.replace("Point(", "").replace(")", "")
    lon, lat = clean.split(" ")

    lon = float(lon)
    lat = float(lat)

    is_valid = -180 <= lon <= 180 and -90 <= lat <= 90

    rows.append((port_id, lon, lat, is_valid))

    if not is_valid:
        invalid_rows.append((port_id, lon, lat))

total_ports_with_coordinates = len(rows)
invalid_coordinates = len(invalid_rows)
valid_coordinates = total_ports_with_coordinates - invalid_coordinates
percent_valid_coordinates = (
    valid_coordinates / total_ports_with_coordinates * 100
    if total_ports_with_coordinates else 0
)

print("\n=== GEOSPATIAL VALIDATION ===\n")
print(f"PORTS WITH COORDINATES: {total_ports_with_coordinates:,}")
print(f"VALID COORDINATES: {valid_coordinates:,}")
print(f"INVALID COORDINATES: {invalid_coordinates:,}")
print(f"PERCENT VALID COORDINATES: {percent_valid_coordinates:.3f}")

with open("geospatial_validation.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow([
        "PortsWithCoordinates",
        "ValidCoordinates",
        "InvalidCoordinates",
        "PercentValidCoordinates"
    ])
    writer.writerow([
        total_ports_with_coordinates,
        valid_coordinates,
        invalid_coordinates,
        percent_valid_coordinates
    ])

with open("invalid_port_coordinates.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow(["Port", "Longitude", "Latitude"])
    writer.writerows(invalid_rows)

print("\nCSV exported: geospatial_validation.csv")
print("CSV exported: invalid_port_coordinates.csv")


=== GEOSPATIAL VALIDATION ===

PORTS WITH COORDINATES: 1,927
VALID COORDINATES: 1,927
INVALID COORDINATES: 0
PERCENT VALID COORDINATES: 100.000

CSV exported: geospatial_validation.csv
CSV exported: invalid_port_coordinates.csv
